# φ2 Hamiltonian Truncation on a Circle Timing Improvement

Builds the truncated Hamiltonian for an interacting scalar field in 1+1D on a circle of radius $r$:

$$H_\text{eff} = H_0 + \frac{m_V^2}{2} H_2$$


and studies how the energy gap $E_1^+ - E_0$ converges as the energy cutoff $E_\mathrm{max}\to\infty$.

Reference: arXiv:2507.15941


In [12]:
import math
import numpy as np
import time
from scipy.special import factorial

%matplotlib inline

## 1. Single-particle energy and mode indexing

Modes are labelled by magnitude $l \ge 0$ (momentum $\pm l/r$) and sign $\sigma = \pm 1$.
States are stored as dictionaries `{...,-l₂:n₂₋,-l₁:n₁₋,l₀:n₀,l₁:n₁₊,l₂:n₂₊,...}`.


In [7]:
def omega(l, m, r):
    """State Frequency"""
    return math.sqrt(float(l)*float(l)/(r*r) + m*m)

def gen_omega_list(lmax, m, r):
    global omega_list
    omega_list = [omega(abs(l), m, r) for l in range(-lmax,lmax+1)]

def gen_omega_list_positive(lmax, m, r):
    global omega_list_positive
    omega_list_positive = [omega(abs(l), m, r) for l in range(0,lmax+1)]
    

#Returns state held in dictionary to list.
def convert_state_to_list(state, lmax):
    return [state.get(i, 0) for i in range(-lmax, lmax+1)]

## Timing Operators

In [22]:
def lower2(stA, stB, mQ, r): 
    #Lowers state A to state B.
    # --- Create a dictionary of the 'difference' between state A and state B.
    diff = stA.copy() 
    for j in stB.keys():
        if j in stA.keys(): diff[j] = (stA[j] - stB[j])
    
    # --- Calculating the normalisation of the state, and applying the operator with the correct symmetry factor.
    norm = 1; factor = 2 #Naive symmetry factor.
    for l in diff.keys():
        factor *= 1/(factorial(diff[l]) * math.sqrt(omega(abs(l),mQ,r)**diff[l]))
        #Iterate through the difference dictionary and update the state norm.
        i=0
        while diff[l]>0:
            norm*=math.sqrt(stA[l]-i)
            i+=1; diff[l]-=1
    return norm*factor

In [3]:
#numpy.where Version.
def lower2V(stA, stB, lmax): 
    
    # --- Create a vector of the 'difference' between state A and state B.
    diff = stA - stB
    
    # --- Calculating the normalisation of the state, and applying the operator with the correct symmetry factor.

    if diff[lmax] == 2: return math.sqrt(stA[lmax]*(stA[lmax]-1)) * 1/omega_list[lmax]
    
    else: 
        return 2*np.prod(np.where(diff==1, np.sqrt(stA) * 1/np.sqrt(omega_list),1))
        

In [4]:
#Iterating over Vector Version.
def lower2VIt(stA, stB, lmax): 
    
    # --- Create a vector of the 'difference' between state A and state B.
    diff = stA - stB
    
    # --- Calculating the normalisation of the state, and applying the operator with the correct symmetry factor.
    res=2 #Naive symmetry factor.

    for i,l in enumerate(range(-lmax,lmax+1)):
        if diff[i]>0:
            if l==0:
                #Remove twice from the same state.
                return math.sqrt(stA[i])*math.sqrt(stA[i]-1) * (1/omega_list[i])
            else:
                #Remove from the same state with opposite momenta.
                res*= math.sqrt(stA[i]) * (1/math.sqrt(omega_list[i]))
                
    return res

In [24]:
#Dictionary Version.
def lower2D(stA, stB): 
    #Lowers state A to state B.
    # --- Create a dictionary of the 'difference' between state A and state B.
    diff = {}
    for j in stA.keys():
        if j in stB.keys(): 
            diff[j] = stA[j] - stB[j]
        else:
            diff[j]=stA[j]

    # --- Calculating the normalisation of the state, and applying the operator with the correct symmetry factor.
    if 0 in diff.keys() and diff[0]>0: 
        return math.sqrt(stA[0]*(stA[0]-1)) * (1/omega_list_positive[0]) 
    else: 
        l=list(diff.keys())[0]
        return 2*math.sqrt(stA[l]*stA[-l]) / (omega_list_positive[abs(l)])


In [43]:
# --- Sample states for timing test:
stA={10:2,0:3,-10:2}; stB = {10:1,0:3,-10:1}; lmaxeff=20
mQ=1; r=1

gen_omega_list_positive(lmaxeff, mQ, r) #For use with dictionary version.
gen_omega_list(lmaxeff, mQ, r) #For use with vector versions.

# --- Methods:
#Dictionary Version Time:
print(f'Old Dictionary Result {lower2(stA, stB, mQ, r)}')
%timeit lower2(stA, stB, mQ, r)
print()

#Dictionary Version Time:
print(f'Dictionary Result {lower2D(stA, stB)}')
%timeit lower2D(stA, stB)
print()

#Convert to numpy array first for fairness
stA = np.array(convert_state_to_list(stA, lmaxeff)); stB = np.array(convert_state_to_list(stB, lmaxeff))

print(f'np.where Result {lower2V(stA, stB, lmaxeff)}')
%timeit lower2V(stA, stB, lmaxeff)
print()
print(f'Iteration Result {lower2VIt(stA, stB, lmaxeff)}')
%timeit lower2VIt(stA, stB, lmaxeff)

Old Dictionary Result 0.39801487608399577
9.41 μs ± 62.4 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)

Dictionary Result 0.39801487608399566
528 ns ± 1.85 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)

np.where Result 0.3980148760839957
5.23 μs ± 24.5 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)

Iteration Result 0.3980148760839957
3.38 μs ± 12.3 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


In [41]:
stA={10:2,0:3,-10:2}; stB = {10:1,0:3,-10:1}; lmaxeff=20
mQ=1; r=1

gen_omega_list_positive(lmaxeff, mQ, r) #For use with dictionary version.
print(lower2D(stA, stB))
%timeit lower2D(stA, stB)


0.39801487608399566
532 ns ± 2.4 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)


In [ ]:
def raise1low1(stA, stB, mQ, r):#Currently set up so energy of state A > energy of state B.
    diff = stA.copy() #Start with state A
    for j in stB.keys():
        if j in stA.keys(): diff[j] = (stA[j] - stB[j])
        elif stB[j]>0:
            diff[j]=-stB[j]
    
    hammDist = sum(map(abs,diff.values()))
    
    
    if hammDist==2:
        norm = 1; factor = 2 #Naive symmetry factor.
        for l in diff.keys():
            factor *= 1/factorial(abs(diff[l])) #Fixes symmetry factor.
            
            i=0
            while diff[l]!=0:
                factor *= 1/math.sqrt(omega(abs(l),mQ,r)) #Energy Scaling for Operator.
                if diff[l]>0:
                    norm*=math.sqrt(stA[l]-i)
                    i+=1
                    diff[l]-=1
                else:
                    if l in stA.keys():
                        norm*=math.sqrt(stA[l]+i+1)
                    else:
                        norm*=math.sqrt(i+1)
                    i+=1
                    diff[l]+=1

        return norm*factor
    

    elif hammDist==0: #Same state...
        norm = 1; factor = 2
        
        tempSum = 0
        #Summing over all ways to do remove and add the same states.
        for l in stA.keys():
            if stA[l]>0:
                #Remove once and add once from this state.
                factorSum = 1/omega(abs(l),mQ,r) #Energy Scaling for Operator.
                normSum = np.sqrt(stA[l])*np.sqrt(stA[l])
                
                tempSum += normSum*factorSum
                
        return norm*factor*tempSum
    
    else: return 0

In [ ]:
def raiselow_hamdistzero(st):
    tempSum = 0
    for l in st.keys():
        if st[l]>0:
            tempSum += st[l] / omega_list_positive[l]